The goal of this notebook is to create the input spectral dataset for external validation of the prioritization model for positive ionization mode. The point is to create an mgf file of filtered positive spectra, unseen to the model. For that reason, GNPS-NIST14-Matches is used.

In [1]:
from matchms.importing import load_from_mgf
from collections import Counter
from matchms.filtering.default_pipelines import DEFAULT_FILTERS
from matchms import SpectrumProcessor
import networkx as nx
from collections import defaultdict
from matchms.similarity import CosineGreedy
from matchms import calculate_scores
from matchms.exporting import save_as_mgf

In [2]:
library = list(load_from_mgf('/home/ioannis/thesis_data/GNPS-NIST14-MATCHES.mgf'))
print('Number of spectra in this library: ', len(library))

Number of spectra in this library:  5763


In [3]:
library[0].metadata

{'charge': 1,
 'ionmode': 'positive',
 'smiles': 'CC(C)C[C@@H](C(=O)O)NC(=O)[C@H](CC1=CC=CC=C1)N',
 'inchi': 'InChI=1S/C15H22N2O3/c1-10(2)8-13(15(19)20)17-14(18)12(16)9-11-6-4-3-5-7-11/h3-7,10,12-13H,8-9,16H2,1-2H3,(H,17,18)(H,19,20)/t12-,13-/m0/s1',
 'scans': '864',
 'ms_level': '2',
 'instrument_type': 'ESI-qTof',
 'file_name': 'MSV000081152/ccms_peak/mzXML_Files/Plate_1/C18p_Plate1_BE11_01_10247.mzXML',
 'peptide_sequence': '*..*',
 'organism_name': 'GNPS-NIST14-MATCHES',
 'compound_name': 'Spectral Match to Phe-Leu from NIST14 M+H',
 'principal_investigator': 'Data from Suryasarathi Dasgupta',
 'data_collector': 'Data deposited by fevargas',
 'submit_user': 'mwang87',
 'confidence': '3',
 'spectrum_id': 'CCMSLIB00003134487',
 'precursor_mz': 279.171}

In [4]:
ionmode = [spectrum.get('ionmode') for spectrum in library]
print(Counter(ionmode))

Counter({'positive': 5277, 'negative': 484, 'positive-20ev': 2})


Since this particular external validation is for the positive ion mode, we isolate the positively charged spectra.

In [5]:
positive_library = [spectrum for spectrum in library if spectrum.get('ionmode') == 'positive']
print('Spectra with positive ionization: ', len(positive_library))

Spectra with positive ionization:  5277


In [6]:
# Further filter to only those with SMILES metadata
positive_with_smiles = [s for s in positive_library if s.get("smiles") is not None]
print("Positive spectra with SMILES:", len(positive_with_smiles))

Positive spectra with SMILES: 4667


This spectral library hasn't been preprocessed. For that reason, we use the preprocessing from matchms that was also used by de Jonge et al. for the MSnLib & GNPS spectral library.

In [7]:
processor = SpectrumProcessor(DEFAULT_FILTERS)

In [8]:
positive_cleaned, _ = processor.process_spectra(positive_with_smiles)
positive_cleaned = [s for s in positive_cleaned if s is not None]

Processing spectra:   0%|          | 0/4667 [00:00<?, ?it/s]

Processing spectra:   1%|▏         | 69/4667 [00:00<00:57, 80.42it/s]

2026-03-11 20:19:50,525:WARNING:matchms:clean_adduct:The charge in the adduct: [M+2H]2+ and the given charge: 1 do not match


Processing spectra:   2%|▏         | 114/4667 [00:01<00:53, 84.53it/s]

2026-03-11 20:19:51,050:WARNING:matchms:clean_adduct:The charge in the adduct: [M+2H]2+ and the given charge: 1 do not match


...

Processing spectra:  97%|█████████▋| 4523/4667 [00:55<00:01, 74.48it/s]

2026-03-11 20:20:45,264:WARNING:matchms:clean_adduct:The charge in the adduct: [M+2H]2+ and the given charge: 1 do not match


Processing spectra:  98%|█████████▊| 4568/4667 [00:56<00:01, 85.17it/s]

2026-03-11 20:20:45,774:WARNING:matchms:clean_adduct:The charge in the adduct: [M+2H]2+ and the given charge: 1 do not match


Processing spectra: 100%|██████████| 4667/4667 [00:57<00:00, 81.35it/s]


In [8]:
print(len(positive_cleaned))

4667


In [11]:
from matchms.exporting import save_as_mgf
save_as_mgf(positive_cleaned, "/home/ioannis/thesis_data/testing_positive_cleaned.mgf")

dict_keys(['spectra'])


In [9]:
positive_cleaned[0].metadata

{'charge': 1,
 'ionmode': 'positive',
 'smiles': 'CC(C)C[C@@H](C(=O)O)NC(=O)[C@H](CC1=CC=CC=C1)N',
 'inchi': 'InChI=1S/C15H22N2O3/c1-10(2)8-13(15(19)20)17-14(18)12(16)9-11-6-4-3-5-7-11/h3-7,10,12-13H,8-9,16H2,1-2H3,(H,17,18)(H,19,20)/t12-,13-/m0/s1',
 'scans': '864',
 'ms_level': '2',
 'instrument_type': 'ESI-qTof',
 'file_name': 'MSV000081152/ccms_peak/mzXML_Files/Plate_1/C18p_Plate1_BE11_01_10247.mzXML',
 'peptide_sequence': '*..*',
 'organism_name': 'GNPS-NIST14-MATCHES',
 'compound_name': 'Phe-Leu',
 'principal_investigator': 'Data from Suryasarathi Dasgupta',
 'data_collector': 'Data deposited by fevargas',
 'submit_user': 'mwang87',
 'confidence': '3',
 'spectrum_id': 'CCMSLIB00003134487',
 'precursor_mz': 279.171,
 'adduct': '[M+H]+',
 'retention_index': None,
 'retention_time': None,
 'inchikey': 'RFCVXVPWSPOMFJ-STQMWFEESA-N',
 'parent_mass': 278.163724,
 'formula': 'C15H22N2O3'}

Since this is an annotated spectral library, a metadata completeness check is performed to investigate the consistency of the metadata.

In [10]:
def check_completeness(spectra, metadata):
    total = len(spectra)
    counts = Counter()
    for spectrum in spectra:
        for category in metadata:
            info = spectrum.get(category)
            if info is not None and str(info).strip() != "":
                counts[category] += 1

    print(f"Total spectra: {total}")
    for category in metadata:
        completeness = (counts[category] / total * 100) if total else 0
        print(f"{category:16s}: {counts[category]:6d} / {total}  ({completeness:5.2f}%)")

In [11]:
metadata = ['description', 'formula', 'inchi', 'smiles', 'adduct', 'inchikey', 'collision_energy', 'fragmentation_method', 'ms_mass_analyzer']
check_completeness(positive_cleaned, metadata)

Total spectra: 4667
description     :      0 / 4667  ( 0.00%)
formula         :   4666 / 4667  (99.98%)
inchi           :   4666 / 4667  (99.98%)
smiles          :   4666 / 4667  (99.98%)
adduct          :   4667 / 4667  (100.00%)
inchikey        :   4666 / 4667  (99.98%)
collision_energy:      0 / 4667  ( 0.00%)
fragmentation_method:      0 / 4667  ( 0.00%)
ms_mass_analyzer:      0 / 4667  ( 0.00%)


InChIKey and adduct metadata are consistently present. Collision energy metadata are 0% present.

In [12]:
positive_cleaned = [s for s in positive_cleaned if s.get("inchikey")]
print(len(positive_cleaned))

4666


Spectral Deduplication using metadata information:

In [13]:
def filter_redundant_graph(spectra, similarity_threshold=0.99):
    """
    Deduplicate spectra by first grouping on (InChIKey, adduct),
    then clustering within each group using graph-based redundancy filtering.

    Parameters
    ----------
    spectra : list of matchms.Spectrum
        Input spectra to filter.
    similarity_threshold : float, optional
        Cosine similarity cutoff for redundancy (default = 0.99).

    Returns
    -------
    filtered_spectra : list of matchms.Spectrum
        Deduplicated spectra.
    """

    # group spectra by (inchikey, adduct)
    grouped = defaultdict(list)
    for s in spectra:
        key = (s.get("inchikey"), s.get("adduct"))
        grouped[key].append(s)

    filtered = []
    similarity_function = CosineGreedy()

    # cluster within each group one by one
    for group_spectra in grouped.values():
        if len(group_spectra) == 1:
            filtered.extend(group_spectra)
            continue

        # compute all-vs-all similarities inside the group
        scores = calculate_scores(group_spectra, group_spectra, similarity_function)
        scores_array = scores.to_array(name="CosineGreedy_score")

        # Build graph
        G = nx.Graph()
        G.add_nodes_from(range(len(group_spectra))) # each spectra in the group becomes a node
        for i in range(len(group_spectra)):
            for j in range(i+1, len(group_spectra)):
                if scores_array[i, j] >= similarity_threshold:
                    G.add_edge(i, j)

        # Find connected components (connected nodes in the graph)
        clusters = list(nx.connected_components(G))

        # Pick representative per cluster
        for cluster in clusters:
            best = max(
                cluster,
                key=lambda idx: float(group_spectra[idx].get("quality_explained_intensity", 0))
            )
            filtered.append(group_spectra[best])

    print(f"Original spectra: {len(spectra)}")
    print(f"Filtered spectra: {len(filtered)}")
    return filtered


In [14]:
filtered_cleaned_positive = filter_redundant_graph(positive_cleaned)

Original spectra: 4666
Filtered spectra: 3512


In [15]:
save_as_mgf(filtered_cleaned_positive, "/home/ioannis/thesis_data/GNPS-NIST14-MATCHES_pos_cleaned_deduplicated.mgf")

dict_keys(['spectra'])
